In [19]:
import pandas as pd
import numpy as np

In [18]:
url = "https://raw.githubusercontent.com/EveliaCoss/CAMDA2025_metadatos/refs/heads/main/rawdata/TrainAndTest_cleaned/training_metadata_cleaned.tsv"
df = pd.read_csv(url, sep='\t')

In [2]:
df.to_csv("training_metadata_cleaned.csv", index=False)

In [17]:
def process_MIC_data(input_file, output_file):
    """
    Procesa un archivo CSV con datos MIC (Concentración Mínima Inhibitoria),
    recategoriza los valores MIC y asigna un fenotipo (Susceptible o Resistente)
    basado en tablas interpretativas específicas para cada género.

    Parámetros:
    -----------
    input_file : str
        Ruta al archivo CSV de entrada que contiene las columnas 'measurement_value', 'accession' y 'new_genus'.

    output_file : str
        Ruta del archivo CSV donde se guardará el DataFrame procesado.

    Retorna:
    --------
    df : pandas.DataFrame
        El DataFrame resultante con las columnas nuevas 'recategorized_MIC' y 'phenotype_assigned'.
    """

    df = pd.read_csv(input_file)
    
    # Asegurarse de que measurement_value sea numérico; valores no convertibles se vuelven NaN
    df["measurement_value"] = pd.to_numeric(df["measurement_value"], errors="coerce")

    # Definir intervalos (bins) y etiquetas para categorizar valores de MIC
    bins = [0, 0.09, 0.185, 0.375, 0.75, 1.5, 3, 6, 12, 24, 48, 10000]
    labels = [0.06, 0.12, 0.25, 0.5, 1, 2, 4, 8, 16, 32, 64]

    # Eliminar registros con ciertos códigos de acceso no deseados
    accessions_to_remove = [
        "ERR1218638", "ERR1218722", "SRR2101499", "SRR960879",
        "SRR850995", "ERR1218771", "SRR5386043", "SRR6985679"
    ]
    df = df[~df["accession"].isin(accessions_to_remove)]

    # Categorizar los valores de MIC en rangos definidos
    recategorized_MIC = pd.cut(df["measurement_value"], bins=bins, labels=labels, right=False)

    df.insert(df.columns.get_loc("measurement_value") + 1, "recategorized_MIC", recategorized_MIC)

    # Tabla de interpretación de fenotipos para distintos géneros bacterianos
    phenotype_table = {
        "Klebsiella":              list('sssssssrrrr'),
        "Escherichia":             list('sssssssrrrr'),
        "Salmonella":              list('sssssssrrrr'),
        "Streptococcus":           list('ssssrrrrrrr'),
        "Staphylococcus":          list('sssssssrrrr'),
        "Pseudomonas":             list('sssssssssrr'),
        "Acinetobacter":           list('ssssssssrrr'),
        "Campylobacter":           list('ssssssssrrr'),
        "Neisseria":               list('sssssrrrrrr')
    }

    # Convertir la tabla de interpretación a DataFrame, con índices por genus y columnas por MIC
    phenotype_df = pd.DataFrame(phenotype_table, index=labels).T

    # Función auxiliar para asignar fenotipo a cada fila según el genus y el MIC recategorizado
    def assign_phenotype(row):
        genus = row["new_genus"]
        mic = row["recategorized_MIC"]
        if pd.isna(genus) or pd.isna(mic):
            return np.nan
        if genus in phenotype_df.index:
            return "Susceptible" if phenotype_df.loc[genus, mic] == "s" else "Resistant"
        else:
            return np.nan

    # Aplicar la función a cada fila y crear la nueva columna
    phenotype_assigned = df.apply(assign_phenotype, axis=1)

    # Insertar la columna phenotype_assigned 
    df.insert(df.columns.get_loc("new_genus") + 1, "phenotype_assigned", phenotype_assigned)
    df.to_csv(output_file, index=False)
    return df


In [11]:
df_resultado = process_MIC_data(
    input_file="training_metadata_cleaned.csv",
    output_file="CAMDA25_training_con_MIC_y_fenotipo.csv"
)


In [13]:
iguales = (df_resultado['phenotype'] == df_resultado['phenotype_assigned']).sum()
distintas = (df_resultado['phenotype'] != df_resultado['phenotype_assigned']).sum()
total = len(df_resultado)

print(f"Total de filas: {total}")
print(f"Filas donde genus = new_genus: {iguales}")
print(f"Filas donde genus ≠ new_genus: {distintas}")

Total de filas: 5715
Filas donde genus = new_genus: 4267
Filas donde genus ≠ new_genus: 1448


In [15]:
diferencias = df_resultado[df_resultado["phenotype"] != df_resultado["phenotype_assigned"]]
diferencias


,genus,species,accession,genome,ani,new_genus,phenotype_assigned,phenotype,antibiotic,measurement_sign,...,testing_standard_year,publication,isolation_source,isolation_country,collection_date,scientific_name_CAMDA,scientific_name_NCBI,scientific_name_Antibiogram,status,status_reference
13,Neisseria,gonorrhoeae,ERR388428,NaN,NaN,NaN,NaN,Susceptible,TET,NaN,...,2013,31358980,NaN,China,NaN,Neisseria gonorrhoeae,NaN,NaN,Missing_All,No_sources
18,Neisseria,gonorrhoeae,ERR350006,NaN,NaN,NaN,NaN,Susceptible,TET,NaN,...,2013,31358980,NaN,Slovenia,NaN,Neisseria gonorrhoeae,NaN,NaN,Missing_All,No_sources
19,Neisseria,gonorrhoeae,ERR350033,NaN,NaN,NaN,NaN,Susceptible,TET,NaN,...,2013,31358980,NaN,Slovenia,NaN,Neisseria gonorrhoeae,Neisseria gonorrhoeae,NaN,Matched_NCBI,Verify_sources
52,Neisseria,gonorrhoeae,ERR350060,NaN,NaN,NaN,NaN,Susceptible,TET,NaN,...,2013,31358980,NaN,Poland,NaN,Neisseria gonorrhoeae,NaN,NaN,Missing_All,No_sources
63,Neisseria,gonorrhoeae,ERR352811,NaN,NaN,NaN,NaN,Susceptible,TET,NaN,...,2013,31358980,NaN,Thailand,NaN,Neisseria gonorrhoeae,NaN,NaN,Missing_All,No_sources
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5715,Pseudomonas,aeruginosa,SRR8737535,ENA_SAMN11110345,0.993,Pseudomonas,Susceptible,Intermediate,CAZ,NaN,...,2013,32048461,NaN,Germany,NaN,Pseudomonas aeruginosa,Pseudomonas aeruginosa,NaN,Matched_NCBI,Verify_sources
5716,Pseudomonas,aeruginosa,SRR8737453,BVBRC_287.12443,0.989,Pseudomonas,Susceptible,Intermediate,CAZ,NaN,...,2013,32048461,NaN,Germany,NaN,Pseudomonas aeruginosa,Pseudomonas aeruginosa,Pseudomonas aeruginosa,Matched_NCBI_Antibiogram,Good_sources
5717,Pseudomonas,aeruginosa,SRR5642504,ENA_SAMN07185694,0.993,Pseudomonas,Susceptible,Intermediate,CAZ,NaN,...,2013,NaN,blood,USA,NaN,Pseudomonas aeruginosa,Pseudomonas aeruginosa,Pseudomonas aeruginosa,Matched_NCBI_Antibiogram,Good_sources
5718,Pseudomonas,aeruginosa,SRR8737597,BVBRC_287.12495,0.993,Pseudomonas,Susceptible,Intermediate,CAZ,NaN,...,2013,32048461,NaN,Germany,NaN,Pseudomonas aeruginosa,Pseudomonas aeruginosa,Pseudomonas aeruginosa,Matched_NCBI_Antibiogram,Good_sources
